# 📓 Notebook 05 — Fluxo Contínuo de Drift e Agentes Orquestradores

> **Pergunta operacional:** como manter o pipeline vivo, com detecção de drift contínua e um **agente de IA** decidindo a ação?
>
> **Origem canônica:** Disciplina 01, Aula 05 (monitoramento, CI/CT, drift, playbooks).

## Objetivo

Fechar o ciclo da Etapa 1 do TC mostrando o **ponto onde a Etapa 1 vira Etapa 4**: detector contínuo de drift + agente orquestrador.

1. Simular um stream de produção com drift gradual;
2. Calcular PSI / KS por janela e disparar alertas;
3. Conectar os alertas a um **agente** com loop ReAct simples que escolhe ações (`hold`, `alert`, `partial_retrain`, `full_retrain`);
4. Registrar todas as decisões em um ledger auditável.

## Por que isso importa

Modelo entregue na Fase 01 será observado nas Fases 04 e 05. Sem detector contínuo + agente, o Tech Challenge entrega um modelo que **degrada silenciosamente**.

## 0. Setup

In [1]:
from __future__ import annotations

import json
from dataclasses import asdict, dataclass, field
from datetime import datetime, timedelta
from pathlib import Path
from typing import Any, Callable, Iterator

import numpy as np
import pandas as pd
from scipy.stats import ks_2samp
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_SEED = 42
RNG = np.random.default_rng(RANDOM_SEED)

df = pd.read_csv(Path("../dataset/processed/telco_churn.csv"))
df["churn_bin"] = (df["churn"] == "Yes").astype(int)

FEATURES = [
    "tenure",
    "monthly_charges",
    "total_charges",
    "contract",
    "internet_service",
    "payment_method",
]
NUM = ["tenure", "monthly_charges", "total_charges"]
CAT = [c for c in FEATURES if c not in NUM]
TARGET = "churn_bin"

## 1. Treino do modelo "de referência"

Tomamos 70% dos dados como produção-estável e treinamos o modelo. O restante simula a sequência temporal de inferências em produção.

In [2]:
from sklearn.model_selection import train_test_split

ref_df, stream_df = train_test_split(df, test_size=0.30, stratify=df[TARGET], random_state=RANDOM_SEED)

preprocess = ColumnTransformer(
    [
        ("num", StandardScaler(), NUM),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CAT),
    ]
)
model = Pipeline([("prep", preprocess), ("clf", LogisticRegression(max_iter=500, random_state=RANDOM_SEED))])
model.fit(ref_df[FEATURES], ref_df[TARGET])

ref_auc = roc_auc_score(ref_df[TARGET], model.predict_proba(ref_df[FEATURES])[:, 1])
print(f"AUC na referência: {ref_auc:.3f}")

AUC na referência: 0.967


## 2. Simulando drift gradual

Injetamos três regimes ao longo do tempo na janela de produção:

1. **Estável** (semanas 1–4) — distribuição original.
2. **Drift de cobertura** (semanas 5–8) — `monthly_charges` sobe 10% (campanha de reajuste).
3. **Drift de conceito** (semanas 9–12) — `Month-to-month` mais propenso a churn (cliente mais sensível a preço).

In [3]:
stream_df = stream_df.reset_index(drop=True)
n = len(stream_df)
weeks = np.linspace(1, 12, n).astype(int)
stream_df["week"] = weeks

# Drift 2: reajuste de 10% nos meses 5-8.
mask_2 = stream_df["week"].between(5, 8)
stream_df.loc[mask_2, "monthly_charges"] = stream_df.loc[mask_2, "monthly_charges"] * 1.10

# Drift 3: aumenta churn em Month-to-month nas semanas 9-12.
mask_3 = stream_df["week"].between(9, 12) & (stream_df["contract"] == "Month-to-month")
flip_idx = stream_df[mask_3].sample(frac=0.25, random_state=RANDOM_SEED).index
stream_df.loc[flip_idx, "churn_bin"] = 1

stream_df.groupby("week")[["monthly_charges", "churn_bin"]].mean().round(3)

,monthly_charges,churn_bin
week,,
1,67.654,0.247
2,70.485,0.280
3,64.406,0.271
4,71.835,0.239
5,75.223,0.261
6,77.922,0.289
7,74.523,0.257
8,73.674,0.289
9,64.054,0.390


## 3. Detector contínuo: PSI + KS + AUC corrente

Cada janela semanal gera três sinais. O detector decide a severidade e empacota um evento para o agente.

In [4]:
@dataclass(slots=True)
class DriftSignal:
    week: int
    psi: float
    ks_pvalue: float
    auc: float
    severity: str


def psi(expected: np.ndarray, actual: np.ndarray, bins: int = 10) -> float:
    quantiles = np.unique(np.quantile(expected, np.linspace(0, 1, bins + 1)))
    if len(quantiles) < 3:
        return 0.0
    exp_hist, _ = np.histogram(expected, bins=quantiles)
    act_hist, _ = np.histogram(actual, bins=quantiles)
    exp_pct = np.clip(exp_hist / exp_hist.sum(), 1e-6, None)
    act_pct = np.clip(act_hist / max(act_hist.sum(), 1), 1e-6, None)
    return float(np.sum((act_pct - exp_pct) * np.log(act_pct / exp_pct)))


def classify(psi_value: float, ks_pvalue: float, auc: float) -> str:
    if auc < 0.65 or psi_value > 0.25:
        return "high"
    if psi_value > 0.1 or ks_pvalue < 0.01 or auc < 0.75:
        return "medium"
    return "low"


def stream_signals() -> Iterator[DriftSignal]:
    ref_mc = ref_df["monthly_charges"].to_numpy()
    for week, chunk in stream_df.groupby("week"):
        proba = model.predict_proba(chunk[FEATURES])[:, 1]
        auc = float(roc_auc_score(chunk[TARGET], proba)) if chunk[TARGET].nunique() == 2 else float("nan")
        psi_value = psi(ref_mc, chunk["monthly_charges"].to_numpy())
        ks = ks_2samp(ref_mc, chunk["monthly_charges"].to_numpy())
        yield DriftSignal(
            week=int(week),
            psi=round(psi_value, 3),
            ks_pvalue=round(float(ks.pvalue), 4),
            auc=round(auc, 3),
            severity=classify(psi_value, float(ks.pvalue), auc),
        )

signals = list(stream_signals())
pd.DataFrame([asdict(s) for s in signals])

,week,psi,ks_pvalue,auc,severity
0,1,0.050,0.7951,0.969,low
1,2,0.020,0.7332,0.935,low
2,3,0.033,0.0857,0.956,low
3,4,0.029,0.2174,0.942,low
4,5,0.106,0.0110,0.954,medium
5,6,0.106,0.0042,0.953,medium
6,7,0.042,0.1003,0.956,low
7,8,0.070,0.0329,0.975,low
8,9,0.043,0.0249,0.931,low
9,10,0.045,0.5852,0.919,low


## 4. Agente orquestrador

Um agente recebe os sinais, raciocina sobre o estado e escolhe uma ação. Aqui usamos um **loop ReAct determinístico** (sem LLM real) para o material rodar sem dependência de chaves. A interface `complete()` aceita um cliente compatível com OpenAI/Foundry caso a equipe queira plugar um LLM depois.

In [5]:
ActionFn = Callable[["AgentContext"], dict[str, Any]]


@dataclass(slots=True)
class AgentContext:
    signal: DriftSignal
    last_action: str = "none"
    consecutive_high: int = 0


@dataclass(slots=True)
class Decision:
    week: int
    severity: str
    action: str
    rationale: str
    payload: dict[str, Any] = field(default_factory=dict)


def action_hold(ctx: AgentContext) -> dict[str, Any]:
    return {"name": "hold", "rationale": "sinais dentro do envelope; manter modelo atual."}


def action_alert(ctx: AgentContext) -> dict[str, Any]:
    return {"name": "alert", "rationale": f"PSI={ctx.signal.psi} no limite; abrir incidente nível 3."}


def action_partial_retrain(ctx: AgentContext) -> dict[str, Any]:
    return {
        "name": "partial_retrain",
        "rationale": f"AUC={ctx.signal.auc} baixou; reusar pipeline e re-treinar nas últimas 4 semanas.",
    }


def action_full_retrain(ctx: AgentContext) -> dict[str, Any]:
    return {
        "name": "full_retrain",
        "rationale": "dois episódios severos consecutivos; rotina completa de re-treino + revisão de schema.",
    }


TOOLS: dict[str, ActionFn] = {
    "hold": action_hold,
    "alert": action_alert,
    "partial_retrain": action_partial_retrain,
    "full_retrain": action_full_retrain,
}


def policy(ctx: AgentContext) -> str:
    """Política determinística — substituível por LLM compatível com OpenAI."""
    sev = ctx.signal.severity
    if sev == "low":
        return "hold"
    if sev == "medium":
        return "alert"
    if ctx.consecutive_high >= 1:
        return "full_retrain"
    return "partial_retrain"


def run_agent(signals: list[DriftSignal]) -> list[Decision]:
    ctx = AgentContext(signal=signals[0])
    log: list[Decision] = []
    for sig in signals:
        ctx.signal = sig
        ctx.consecutive_high = ctx.consecutive_high + 1 if sig.severity == "high" else 0
        action_name = policy(ctx)
        result = TOOLS[action_name](ctx)
        log.append(
            Decision(
                week=sig.week,
                severity=sig.severity,
                action=result["name"],
                rationale=result["rationale"],
                payload={"psi": sig.psi, "auc": sig.auc, "ks_pvalue": sig.ks_pvalue},
            )
        )
        ctx.last_action = action_name
    return log


decisions = run_agent(signals)
pd.DataFrame([asdict(d) for d in decisions])

,week,severity,action,rationale,payload
0,1,low,hold,sinais dentro do envelope; manter modelo atual.,"{'psi': 0.05, 'auc': 0.969, 'ks_pvalue': 0.7951}"
1,2,low,hold,sinais dentro do envelope; manter modelo atual.,"{'psi': 0.02, 'auc': 0.935, 'ks_pvalue': 0.7332}"
2,3,low,hold,sinais dentro do envelope; manter modelo atual.,"{'psi': 0.033, 'auc': 0.956, 'ks_pvalue': 0.0857}"
3,4,low,hold,sinais dentro do envelope; manter modelo atual.,"{'psi': 0.029, 'auc': 0.942, 'ks_pvalue': 0.2174}"
4,5,medium,alert,PSI=0.106 no limite; abrir incidente nível 3.,"{'psi': 0.106, 'auc': 0.954, 'ks_pvalue': 0.011}"
5,6,medium,alert,PSI=0.106 no limite; abrir incidente nível 3.,"{'psi': 0.106, 'auc': 0.953, 'ks_pvalue': 0.0042}"
6,7,low,hold,sinais dentro do envelope; manter modelo atual.,"{'psi': 0.042, 'auc': 0.956, 'ks_pvalue': 0.1003}"
7,8,low,hold,sinais dentro do envelope; manter modelo atual.,"{'psi': 0.07, 'auc': 0.975, 'ks_pvalue': 0.0329}"
8,9,low,hold,sinais dentro do envelope; manter modelo atual.,"{'psi': 0.043, 'auc': 0.931, 'ks_pvalue': 0.0249}"
9,10,low,hold,sinais dentro do envelope; manter modelo atual.,"{'psi': 0.045, 'auc': 0.919, 'ks_pvalue': 0.5852}"


### 🛑 Breakpoint — discussão

1. Qual a diferença entre `partial_retrain` e `full_retrain` em termos de risco?
2. Quando vale ter um humano no loop (HITL) entre o sinal e a ação?
3. Como mudaria o agente se cada decisão custasse R$ 10 mil em pipeline computacional?

## 5. Ledger auditável

Todo agente em produção precisa de um log que sobreviva ao processo. Aqui salvamos JSON-lines — em produção, troca-se por Event Hub, Service Bus ou Kafka.

In [6]:
ledger_path = Path("../dataset/processed/agent_ledger.jsonl")
with ledger_path.open("w", encoding="utf-8") as fh:
    base_ts = datetime(2026, 1, 1)
    for d in decisions:
        record = {
            "timestamp": (base_ts + timedelta(weeks=d.week - 1)).isoformat(),
            "week": d.week,
            "severity": d.severity,
            "action": d.action,
            "rationale": d.rationale,
            "payload": d.payload,
        }
        fh.write(json.dumps(record, ensure_ascii=False) + "\n")
print(f"Ledger gravado em {ledger_path}")

Ledger gravado em ..\dataset\processed\agent_ledger.jsonl


## 6. Plugando um LLM compatível com OpenAI (stub)

A `policy()` é determinística para fins de aula. Para subir um agente real (Foundry, Azure OpenAI, Bedrock), basta substituir por uma chamada como o stub abaixo. **O bloco está comentado** para não exigir credenciais.

In [7]:
stub_prompt_template = '''
Você é um agente operacional de MLOps. Receba sinais de drift e escolha UMA ação dentre:
{tools}

Sinais:
{signal_json}
Histórico:
{history_json}

Responda em JSON: {{"action": "<nome>", "rationale": "<motivo curto>"}}.
'''
print("Prompt template pronto. Para ativar com um cliente OpenAI/Foundry:")
print("client.chat.completions.create(model=..., messages=[...]) ➜ parse JSON ➜ TOOLS[result['action']](ctx)")

Prompt template pronto. Para ativar com um cliente OpenAI/Foundry:
client.chat.completions.create(model=..., messages=[...]) ➜ parse JSON ➜ TOOLS[result['action']](ctx)


## 🧠 Exercícios

**Iniciante.** Adicione PSI por `payment_method` ao detector. O que muda nos sinais?

**Intermediário.** Troque `policy()` por uma função baseada em LLM (Azure OpenAI ou Foundry). Como você protege o sistema contra alucinações que escolhem ações inválidas?

**Avançado.** Inclua um plano de **rollback** no agente: se `full_retrain` for executado mas a AUC piorar, restaurar o modelo anterior. Registre no ledger e desenhe a janela de avaliação (em quantas semanas decidimos manter o novo modelo?).

## ✅ Fechando a Etapa 1 do Tech Challenge

Saímos com:

- contrato de dados versionado (NB1);
- separação medalhão × pipeline com CV correta (NB2);
- treino batch / mini-batch / online auditados (NB3);
- Model Card sketch com slice metrics (NB4);
- detector + agente + ledger auditável (NB5).

Próximo encontro: **Etapa 2 do TC — Modelagem com Redes Neurais**, em [`encontro-02`](../../encontro-02/README.md).